# 01｜语料与分词器

130U-FinLLM｜130U 金融大语言模型

本页处理本地已取得并审核的数据。将文件从 Drive 复制到 /content 后再处理。不要直接加载整个大型语料库，也不要把未知授权的社交媒体内容加入训练。具体格式见 docs/DATA_GUIDE.md。先准备任务评测原文池并划分，再清洗和排除预训练污染；最后训练 tokenizer 与打包。

每个 Notebook 都包含独立环境入口。Colab 菜单：运行时 → 更改运行时类型 → T4 GPU。点击执行代码前可查看源码；所有输出均为实际执行后产生，仓库不预填训练成绩。

In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path

REPOSITORY = "https://github.com/130U/130U-FinLLM.git"
REF = "main"  # 首次解析并显示提交；复现时替换成完整的 40 位提交 SHA。
PROJECT = Path("/content/130U-FinLLM")

def command(*args, cwd=None):
    # Explicitly relay subprocess output into the notebook's captured stdout.
    with subprocess.Popen([str(x) for x in args], cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, process.args)

if len(REF) == 40 and all(c in "0123456789abcdefABCDEF" for c in REF):
    REVISION = REF.lower()
else:
    refs = subprocess.check_output(["git", "ls-remote", REPOSITORY, REF], text=True).strip().splitlines()
    if len(refs) != 1:
        raise RuntimeError("REF 不能唯一解析；请填写完整提交 SHA。")
    REVISION = refs[0].split()[0]

if not PROJECT.exists():
    command("git", "clone", "--no-checkout", REPOSITORY, PROJECT)
else:
    remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=PROJECT, text=True).strip()
    if remote.rstrip("/").removesuffix(".git") != REPOSITORY.removesuffix(".git"):
        raise RuntimeError("/content/130U-FinLLM 已属于其他仓库，请选择新目录。")
command("git", "fetch", "--depth", "1", "origin", REVISION, cwd=PROJECT)
command("git", "checkout", "--detach", REVISION, cwd=PROJECT)
os.chdir(PROJECT)
print("本次固定代码提交：", REVISION)

# 保留 Colab 自带的 CUDA / PyTorch；此处仅安装项目的其他依赖。
import torch
if tuple(int(v) for v in torch.__version__.split("+")[0].split(".")[:2]) < (2, 5):
    raise RuntimeError("需要 PyTorch >= 2.5；请更换较新的 Colab 运行环境。")
command(sys.executable, "-m", "pip", "install", "-q", "numpy>=1.26,<3", "tokenizers>=0.21,<0.23", "scikit-learn>=1.5,<2")
command(sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps")

def finllm(*args):
    command(sys.executable, "-u", "-m", "finllm", *args, cwd=PROJECT)

def require(*paths):
    missing = [str(p) for p in paths if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("缺少阶段输入，请先准备或从 Drive 恢复：\n" + "\n".join(missing))

finllm("environment")
disk = shutil.disk_usage("/content")
print(f"/content 剩余磁盘：{disk.free / 2**30:.2f} GiB")
Path("runtime_revision.txt").write_text(REVISION + "\n")

In [ ]:
# 可选持久化：需要你在 Google 的授权界面完成登录。
MOUNT_DRIVE = False  # 需要备份 / 恢复时设为 True，再执行此单元格。
DRIVE_ROOT = Path("/content/drive/MyDrive/130U-FinLLM")
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print("Drive 项目目录：", DRIVE_ROOT)
else:
    print("尚未挂载 Drive；/content 是临时磁盘。正式训练前请启用备份。")

In [ ]:
# 新 Notebook 可能分配新的虚拟机；按需恢复前一阶段的文件。
RESTORE_FROM_DRIVE = False
LIVE_STAGES_TO_RESTORE = []  # 中断恢复时填写，例如 ["pretrain"]；会覆盖本地对应阶段。
if RESTORE_FROM_DRIVE:
    if not MOUNT_DRIVE:
        raise RuntimeError("请先把 MOUNT_DRIVE 设为 True 并完成授权。")
    for folder in ("data", "tokenizer", "runs"):
        source = DRIVE_ROOT / folder
        if source.exists():
            finllm("sync", "--source", source, "--destination", PROJECT / folder, "--working-copy")
    for stage in LIVE_STAGES_TO_RESTORE:
        if stage not in ("pretrain", "domain", "sft", "dpo"):
            raise ValueError("未知检查点阶段：" + str(stage))
        source = DRIVE_ROOT / "live_checkpoints" / stage
        if not source.exists():
            raise FileNotFoundError("缺少阶段自动备份：" + str(source))
        finllm("sync", "--source", source, "--destination", PROJECT / "runs" / stage, "--working-copy")
    print("恢复完成。自动检查点只用于匹配原配置、tokenizer 和数据的继续训练；请指定 RESUME_CHECKPOINT。")

下列输入路径是本项目约定。请先填写每条记录的来源、许可、目标和分组信息。原始文件不会由本页自动获取。正式配置的权限与分区校验失败时，先修复材料；不要绕开检查。

In [ ]:
TASK_INPUT = "data/raw/task.jsonl"
require(TASK_INPUT)
finllm("prepare-task", "--input", TASK_INPUT, "--output", "data/task")

准备获准使用的通用和金融训练文本，并用冻结评测原文池排除污染。tokenizer_train.jsonl 应为两类**训练分区**的代表性混合，不包含 development、calibration 或 test。下面的命令只处理已准备的文件。

In [ ]:
TOKENIZER_INPUT = "data/raw/tokenizer_train.jsonl"
GENERAL_INPUT = "data/raw/general_train.jsonl"
FINANCE_INPUT = "data/raw/finance_train.jsonl"
GENERAL_VALIDATION = "data/raw/general_validation.jsonl"
FINANCE_VALIDATION = "data/raw/finance_validation.jsonl"
FROZEN_EVAL = "data/task/frozen_eval.jsonl"
FROZEN_EXCLUSIONS = "data/task/frozen_exclusions.jsonl"
require(TOKENIZER_INPUT, GENERAL_INPUT, FINANCE_INPUT, GENERAL_VALIDATION, FINANCE_VALIDATION, FROZEN_EVAL)
finllm("freeze-exclusions", "--input", FROZEN_EVAL, "--input", GENERAL_VALIDATION, "--input", FINANCE_VALIDATION, "--output", FROZEN_EXCLUSIONS)
finllm("train-tokenizer", "--input", TOKENIZER_INPUT, "--output", "tokenizer/english", "--vocab-size", "8192", "--exclude", FROZEN_EXCLUSIONS)
finllm("pack", "--input", GENERAL_INPUT, "--tokenizer", "tokenizer/english", "--output", "data/processed/general", "--exclude", FROZEN_EXCLUSIONS)
finllm("pack", "--input", FINANCE_INPUT, "--tokenizer", "tokenizer/english", "--output", "data/processed/finance", "--exclude", FROZEN_EXCLUSIONS)
finllm("pack", "--input", GENERAL_VALIDATION, "--tokenizer", "tokenizer/english", "--output", "data/processed/general_validation", "--exclude", FROZEN_EVAL, "--partition", "validation")
finllm("pack", "--input", FINANCE_VALIDATION, "--tokenizer", "tokenizer/english", "--output", "data/processed/finance_validation", "--exclude", FROZEN_EVAL, "--partition", "validation")

领域混合按 token 数指定：20M 金融＋5M 通用。若源分片小于预算，混合器明确记录循环复用次数，不能把重复 token 报成独立新语料。请查看输出清单中的计数。

In [ ]:
finllm("mix-corpora", "--general", "data/processed/general/manifest.json", "--finance", "data/processed/finance/manifest.json", "--general-tokens", "5000000", "--finance-tokens", "20000000", "--output", "data/processed/domain")

In [ ]:
if MOUNT_DRIVE:
    for folder in ("data", "tokenizer", "runs"):
        source = PROJECT / folder
        if source.exists():
            finllm("sync", "--source", source, "--destination", DRIVE_ROOT / folder)
    shutil.copy2(PROJECT / "runtime_revision.txt", DRIVE_ROOT / "runtime_revision.txt")
    print("备份完成。请保留校验清单，下一运行时按清单恢复。")
else:
    print("成果当前仅在临时运行时中；请挂载 Drive 后重新运行备份单元格。")